# Tokenization & Embeddings- How AI Reads Text

**Module 4 · Lesson 4.1**  
*Netsetos GenAI Engineering Course v2.0*

**In this lesson you will:**
- Byte-Pair Encoding (BPE) - Build a Tokenizer From Scratch
- Compare Tokenizers Across Models
- Token Economics - How Tokenization Affects Your Bill
- Three Tokenizer Families - BPE vs WordPiece vs SentencePiece
- From Tokens to Embeddings - Meaning in Numbers
- Why LLMs Can't Count Letters - Tokenization Quirks

---

> This notebook is generated from the published lesson HTML. The HTML is the source of truth - do not hand-edit this notebook. Re-run the generator if the lesson updates.


In [ ]:
# Install dependencies (if running in Colab or fresh environment)
!pip install tiktoken google-genai numpy -q

# Reproducibility - the lesson uses seeded random values throughout
import numpy as np
np.random.seed(42)

## Why tokenization is the hidden lever behind cost, context, and quirks

Before a model can look "smart", it has to turn your text into numbers. That one step - **tokenization** - quietly sets your API bill, your context budget, and a whole class of bizarre failures (yes, including "how many r's in strawberry"). Then those token IDs become **embeddings**: vectors that carry meaning. This lesson builds both halves from scratch, in code.

> ✂️ **Analogy**
>
> **Tokenization is how the AI cuts a sentence into bite-sized pieces.** You eat a dosa by tearing it into pieces - not swallowing it whole. An LLM "eats" text by splitting it into tokens. *How* it splits matters enormously: "Hyderabad" might be 1 token or 4 tokens depending on the tokenizer. More tokens = more cost, less context window, slower generation.

How different models see the same sentence:

English: "Tokenization is the first step in NLP"

Hindi: "टोकनाइज़ेशन एनएलपी का पहला कदम है"

> 📦 **Info**
>
> Why this matters for your wallet
> Same meaning. English = 8 tokens. Hindi = 15 tokens. At roughly $0.30 per million input tokens (Gemini 2.5 Flash, as of 2026), every Hindi prompt costs nearly **2x more** than the same sentence in English. Tokenization directly controls API cost, context-window usage, and generation speed. Understanding it = saving money.

- **Implement** a byte-pair-encoding (BPE) tokenizer from scratch and trace how it merges characters into subwords.

- **Compare** token counts across GPT, Llama, and Gemini tokenizers and explain the multilingual "token tax".

- **Calculate** the API cost of any prompt from its token count, and turn a token ID into a meaning vector (embedding).

- **Explain** why subword tokenization makes a model miscount the letters in "strawberry".

## Warm-up: 60 seconds of recall

Tap each card to check yourself against earlier lessons. Each one is load-bearing for today.

## Why split words at all? The Goldilocks choice

A model cannot store infinite words - it needs a **fixed list** of tokens. There are three obvious ways to chop text into tokens: **whole words**, single **letters**, or something **in between**. Here is why "in between" wins - and it is the whole reason BPE (Step 1) exists.

> 📦 **Info**
>
> Option A: whole words - the impossible toy box
> *Like keeping a separate ready-made LEGO toy for every word that exists.* It breaks in three ways:
> - **The vocabulary explodes.** There is no fixed set of words - names, slang, typos, code, and every language make it effectively endless. You would need millions of entries (the memory cost is in the next box).
> - **The "unknown word" wall.** Any word not on the list becomes a single `[UNK]` - the model literally cannot read a brand-new word, a typo, or a name like "Hyderabad".
> - **No reuse.** "run", "runs", "running", "runner" become four unrelated entries, each learned from scratch - even though they all share "run".

> ℹ️ **Info**
>
> 📊 The math behind "the vocabulary explodes"
> Say we used **1,000,000** whole words. Each word needs a row of numbers (its "meaning"). How much memory is *just* that lookup table? Step by step:
> - **Rows.** 1,000,000 words = `1,000,000` rows in the table.
> - **Numbers per row.** Each word is stored as 512 numbers → `1,000,000 × 512 = 512,000,000` numbers (about half a billion).
> - **Bytes per number.** Each number takes 4 bytes → `512,000,000 × 4 = 2,048,000,000` bytes (about 2 billion).
> - **Turn bytes into GB.** 1 GB ≈ 1,000,000,000 bytes → `2,048,000,000 ÷ 1,000,000,000 ≈ 2 GB`.
> **Result: about 2 GB just to STORE the word list** - before the model has any actual brain. A real 50,000-token vocab is `50,000 × 512 × 4 ≈ 100 MB` - roughly **20x smaller** (because 1,000,000 ÷ 50,000 = 20). A smaller vocabulary also means a smaller, faster final layer. Lighter and quicker.

> 💡 **Info**
>
> Option B: single letters - too many tiny pieces
> *Like paying for everything in 1-rupee coins.* Letters can spell anything (no unknown-word wall, tiny vocabulary) - but one word is about 4-5 letters, so you get 4-5x more tokens, and that single fact breaks four things:
> - **📊 Attention explodes.** Every token must "look at" every other token, so the work grows like a full comparison grid: 5 tokens each checked against all 5 = 5×5 = 25 checks, 10 tokens = 10×10 = 100. So 4-5x more tokens means **16-25x** more compute and memory. (That square growth is "O(n²)".)
> - **📦 The context window shrinks.** A fixed budget of, say, 128,000 token-slots holds about 96,000 words of subword text but only about 25,000 words of letters - a quarter of the document.
> - **🐢 Generation gets slower and costlier.** The model emits one token per step, so 4-5x more pieces = 4-5x more steps - and APIs bill per token, so it is also 4-5x the cost.
> - **🧩 Meaning gets diluted.** "nation" as one piece carries meaning; as `n-a-t-i-o-n` it is smeared across 6 spots the model must reassemble first.
> The twist: letters *would* let a model count the r's in "strawberry" (Step 6) - they are just too expensive to use everywhere.

> ✅ **Info**
>
> Option C: subwords - the smart LEGO set (the winner)
> *Like a normal LEGO set with handy standard pieces.* Common words stay one piece ("the", "ing"); rare words snap together from smaller ones ("un" + "believe" + "able"); any string is buildable; and the box stays small (about 30,000-200,000 pieces). You get an open vocabulary **and** short sequences - the best of both. That is exactly what the `SimpleBPE` you build next does: start from letters, then glue the most common pairs into bigger pieces.

| Approach | Vocab size | Unknown words? | Sequence length |
|---|---|---|---|
| Whole words | millions (too big) | blocked by [UNK] | short |
| Single letters | ~256 (tiny) | none | 4-5x too long |
| Subwords (BPE) | ~30k-200k | none | short |

---

## 🎯 Concept 1: Byte-Pair Encoding (BPE) - Build a Tokenizer From Scratch

### Byte-Pair Encoding (BPE) - Build a Tokenizer From Scratch

The algorithm behind GPT, Llama, and most modern tokenizers.

> 📜 **Analogy**
>
> **BPE is like creating abbreviations.** If you text your friend "btw" instead of "by the way" - you've merged 3 words into 1 token. BPE does this automatically: it finds the most common pair of characters in the training data, merges them into one token, and repeats. After thousands of merges, "the" is one token, "ing" is one token, but rare words like "Hyderabad" get split into pieces.
> **Where the analogy breaks down:** abbreviations like "btw" are chosen by humans for meaning. BPE has no idea what anything means - it merges purely by frequency, which is why it splits a rare name into ugly fragments while keeping a common typo intact.

**The commonest English word is "the".** Watch BPE discover it from plain letters in just **two merges** - no dictionary, just counting which pair of pieces shows up most.

| Word | Start (letters) | Merge 1: t+h → th | Merge 2: th+e → the |
|---|---|---|---|
| the | the | the | the✅ |
| theme | theme | theme | theme |
| there | there | there | there |

Two merges, and the super-common word "the" is now a **single token** - its own "abbreviation". "theme" and "there" reuse that exact `the` piece for free. The `SimpleBPE` below just does this thousands of times on real text.

**📝 `bpe_from_scratch.py`** - *Algorithm*

In [ ]:
# =============================================
# BPE TOKENIZER FROM SCRATCH
# The exact algorithm behind GPT's tokenizer.
# Learn by building it yourself.
# =============================================

from collections import Counter

class SimpleBPE:
    """Byte-Pair Encoding tokenizer - simplified but real."""
    
    def __init__(self):
        self.merges = {}   # pair → merged token
        self.vocab = {}    # token → id
    
    def _get_pairs(self, tokens: list) -> Counter:
        """Count all adjacent pairs."""
        pairs = Counter()
        for i in range(len(tokens) - 1):
            pairs[(tokens[i], tokens[i+1])] += 1
        return pairs
    
    def _merge_pair(self, tokens: list, pair: tuple) -> list:
        """Merge all occurrences of a pair in the token list."""
        merged = []
        i = 0
        while i < len(tokens):
            if i < len(tokens) - 1 and (tokens[i], tokens[i+1]) == pair:
                merged.append(tokens[i] + tokens[i+1])
                i += 2
            else:
                merged.append(tokens[i])
                i += 1
        return merged
    
    def train(self, text: str, num_merges: int = 10):
        """Train BPE: find and merge the most common pairs."""
        # Start: every character is its own token
        tokens = list(text)
        print(f"Start:  {len(set(tokens))} unique chars, {len(tokens)} total tokens")
        
        for step in range(num_merges):
            pairs = self._get_pairs(tokens)
            if not pairs:
                break
            
            # Find the most frequent pair
            best_pair = pairs.most_common(1)[0]
            pair, count = best_pair[0], best_pair[1]
            
            # Merge it
            merged_token = pair[0] + pair[1]
            tokens = self._merge_pair(tokens, pair)
            self.merges[pair] = merged_token
            
            print(f"Step {step+1:2d}: merge '{pair[0]}' + '{pair[1]}' → '{merged_token}' (count={count}, tokens now={len(tokens)})")
        
        # Build vocabulary
        self.vocab = {tok: i for i, tok in enumerate(sorted(set(tokens)))}
        print(f"\nVocab size: {len(self.vocab)} tokens")
        return tokens
    
    def tokenize(self, text: str) -> list:
        """Tokenize text using learned merges."""
        tokens = list(text)
        for pair, merged in self.merges.items():
            tokens = self._merge_pair(tokens, pair)
        return tokens

# ── Train on sample text ──
text = "low lower lowest new newer newest"
print("BPE Training:\n")
bpe = SimpleBPE()
result = bpe.train(text, num_merges=8)

print(f"\nTokenized: {result}")
print(f"\nTest new text:")
print(f"  'lower' → {bpe.tokenize('lower')}")
print(f"  'newest' → {bpe.tokenize('newest')}")

# Output:
# Start:  9 unique chars, 33 total tokens
# Step  1: merge 'w' + 'e' -> 'we' (count=4, tokens now=29)
# Step  2: merge 'l' + 'o' -> 'lo' (count=3, tokens now=26)
# Step  3: merge ' ' + 'n' -> ' n' (count=3, tokens now=23)
# ... (5 more merges) ...
# Step  8: merge ' ne' + 'we' -> ' newe' (count=2, tokens now=12)
# Vocab size: 7 tokens
#   'lower'  -> ['lo', 'we', 'r']      # 5 letters -> 3 subword tokens
#   'newest' -> ['n', 'e', 'we', 'st'] # merges learned from the corpus

- `__init__` - opens two empty notebooks: `self.merges` (the glue-rules it learns, e.g. `t`+`h` → `th`) and `self.vocab` (the final pieces, each with an id).

- `_get_pairs` - scans the tokens and tallies every adjacent pair: the "which two pieces sit next to each other most often?" count. This is the heart of BPE.

- `_merge_pair` - rebuilds the list, gluing **every** occurrence of one winning pair into a single piece (the "staple them together" step from the table above).

- `train` - the engine. Loop `num_merges` times: **count pairs → take the most common → glue it → save the rule.** Exactly the loop you watched build `the`.

- `tokenize` - for brand-new text, it replays the saved merges in order, so `lower` comes out as the same subword pieces it learned in training.

Read the whole class as one recipe: **count → pick the most common pair → glue → repeat.** The two helpers are just "count" and "glue"; `train` runs the recipe, `tokenize` reuses what it learned.

#### 💡 What just happened

⚡ What just happened?
  We built a BPE tokenizer from scratch. Starting with individual characters, it found the most common pair (like "e"+"r"), merged them into one token, or subword unit ("er"), and repeated. After 8 merges, "lower" goes from 5 characters down to a few subword tokens. **This is the exact algorithm behind GPT's tiktoken, Llama 3+'s byte-level BPE, and Gemini's tokenizer. Real tokenizers run 50,000-100,000 merges on billions of words.** **Tradeoff:** a bigger vocabulary (more merges) means fewer tokens per word, but a larger embedding table and more memory - you trade footprint against speed.

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 2: Compare Tokenizers Across Models

### Compare Tokenizers Across Models

Same text, different tokenizers = different token counts. See how GPT, Gemini, and Llama tokenize differently.

The Tokenization Pipeline

**📝 `compare_tokenizers.py pip install`** - *tiktoken*

In [ ]:
# =============================================
# COMPARE TOKENIZERS ACROSS MODELS
# Same text → different token counts.
# pip install tiktoken
# =============================================

import tiktoken

# GPT-4's tokenizer
enc_gpt4 = tiktoken.get_encoding("cl100k_base")     # GPT-4, GPT-3.5
enc_gpt4o = tiktoken.get_encoding("o200k_base")     # GPT-4o

test_texts = [
    ("English",  "Tokenization is the first step in building an LLM."),
    ("Hindi",    "टोकनाइज़ेशन एलएलएम बनाने का पहला कदम है।"),
    ("Telugu",   "టోకెనైజేషన్ LLM నిర్మించడంలో మొదటి దశ."),
    ("Code",     "def fibonacci(n):\n    return n if n <= 1 else fibonacci(n-1) + fibonacci(n-2)"),
    ("JSON",     '{"name": "Netsetos", "students": 10000, "rating": 4.9}'),
    ("Emoji",    "I love AI! 🤖🚀💡 Let's build something amazing!"),
]

print("Token count comparison:\n")
print(f"  {'Language':10s} {'GPT-4':>7s} {'GPT-4o':>8s} {'Diff':>6s}  Text")
print(f"  {'─'*75}")

for lang, text in test_texts:
    gpt4_tokens = enc_gpt4.encode(text)
    gpt4o_tokens = enc_gpt4o.encode(text)
    diff = len(gpt4_tokens) - len(gpt4o_tokens)
    diff_str = f"-{abs(diff)}" if diff > 0 else f"+{abs(diff)}" if diff < 0 else "="
    print(f"  {lang:10s} {len(gpt4_tokens):>5d}   {len(gpt4o_tokens):>5d}   {diff_str:>5s}  {text[:45]}")

# Show actual tokens for English
print(f"\nGPT-4 tokens for English:")
tokens = enc_gpt4.encode(test_texts[0][1])
decoded = [enc_gpt4.decode([t]) for t in tokens]
print(f"  {decoded}")

print(f"\nGPT-4 tokens for Hindi:")
tokens = enc_gpt4.encode(test_texts[1][1])
print(f"  Token count: {len(tokens)} (vs {len(enc_gpt4.encode(test_texts[0][1]))} for English)")
ratio = len(tokens) / len(enc_gpt4.encode(test_texts[0][1]))
print(f"  Hindi uses ~{ratio:.1f}x more tokens → costs ~{ratio:.1f}x more per prompt (cl100k)!")

# Output:
#   Language     GPT-4   GPT-4o   Diff
#   English         12       12      =
#   Hindi           45       15    -30   <- o200k is far better on Devanagari
#   Telugu          63       18    -45
#   Code            24       24      =
#   JSON            23       23      =
#   Emoji           18       15     -3
# Newer tokenizers (o200k) slash the multilingual token tax.

- `tiktoken.get_encoding(...)` - loads a named tokenizer by its vocabulary: `cl100k_base` is GPT-4's, `o200k_base` is GPT-4o's (newer, more multilingual).

- `test_texts` - the same idea in six flavours (English, Hindi, Telugu, code, JSON, emoji), so you compare like with like.

- `.encode(text)` - the call that matters: turns a string into the list of token IDs the model actually reads. `len(...)` of that list is the token count you pay for.

- The loop runs `.encode` with **both** tokenizers and prints the gap - that gap is the "token tax" made visible.

- `.decode([t])` - the reverse: turns one token ID back into its text, so you can SEE the actual chunks (used again in the quirks demo).

The whole script is one idea: **encode the same text with two tokenizers, count, compare.**

#### 💡 What just happened

⚡ What just happened?
  Same meaning in English (8 tokens) vs Hindi (15+ tokens) vs Telugu (18+ tokens). GPT-4o's newer tokenizer (o200k_base) is more efficient than GPT-4's (cl100k_base) for multilingual text. **Key insight: tokenizers are trained mostly on English text, so non-English languages get split into more pieces → more expensive. GPT-4o improved this by training on more multilingual data.**

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

> ℹ️ **Info**
>
> 🤖 What about Claude? The tokenizer you cannot open
> You just saw GPT's exact pieces because OpenAI **open-sources** its tokenizer (`tiktoken`) - you run it offline and watch every split, like `" LLM"` → `[" L", "LM"]`. **Claude is different.** Anthropic does **not** publish a downloadable tokenizer for the current models (Claude 3 and 4), so there is no `pip install` that prints Claude's pieces. You cannot show Claude's `" L" + "LM"` split the way you can for GPT - that data simply is not public.
> Instead, Anthropic gives you a server-side **token-counting API**. You send your text to `client.messages.count_tokens(model="claude-sonnet-4-6", messages=[...])` and it returns `input_tokens` - a single **number, not the pieces**. Perfect for cost math and context-window budgeting; it just will not show you the chunks.
> **The trap:** do not reuse `tiktoken` to estimate a Claude bill. It is OpenAI's tokenizer on a different vocabulary, so it returns the wrong count for Claude - Anthropic notes it undercounts by roughly **15-20%** on plain text (more on code and non-English). For Claude, the count-tokens API is the only accurate source. [Anthropic token-counting docs](https://platform.claude.com/docs/en/build-with-claude/token-counting).

| Question | tiktoken (GPT) | Claude count-tokens API |
|---|---|---|
| Runs where? | offline, on your machine | a server-side API call |
| Shows the pieces? | yes -['st','raw','berry'] | no - count only |
| Shows the token count? | yes | yes (input_tokens) |
| Subword BPE underneath? | Yes for both - "AI" stays one token, "LLM" splits into a few; you just inspect them differently |  |

---

## 🎯 Concept 3: Token Economics - How Tokenization Affects Your Bill

### Token Economics - How Tokenization Affects Your Bill

Every token costs money. Understanding token economics = controlling costs.

**📝 `token_economics.py Cost`** - *Calculator*

In [ ]:
# =============================================
# TOKEN ECONOMICS CALCULATOR
# How much does your prompt actually cost?
# =============================================

import tiktoken

enc = tiktoken.get_encoding("cl100k_base")

# Pricing per 1M tokens (USD), as of 2026 - always check current rates
PRICES = {
    "Gemini 2.5 Flash (input)":   0.30,
    "Gemini 2.5 Flash (output)":  2.50,
    "GPT-5 (input)":              1.25,
    "GPT-5 (output)":            10.00,
    "Claude Sonnet 4.6 (input)":   3.00,
    "Claude Sonnet 4.6 (output)": 15.00,
}

def analyze_cost(text: str, label: str, daily_calls: int = 10000):
    tokens = enc.encode(text)
    n = len(tokens)
    print(f"\n  📊 {label}")
    print(f"  Characters: {len(text):,} | Tokens: {n:,} | Ratio: {len(text)/n:.1f} chars/token")
    print(f"  Cost per call ({daily_calls:,}/day):")
    for model, price in PRICES.items():
        cost_per_call = n * price / 1e6
        daily = cost_per_call * daily_calls
        monthly = daily * 30
        if "input" in model:
            print(f"    {model:30s}: ${cost_per_call:.6f}/call → ₹{monthly*84:.0f}/month")

print("Token Economics:\n")

# Compare: short prompt vs verbose prompt
short = "Summarize: key points only, 3 bullets"
verbose = "I would like you to please kindly summarize the following text. It is important that you provide the key points in a bulleted list format with exactly 3 bullets."

analyze_cost(short, "Short prompt (optimized)")
analyze_cost(verbose, "Verbose prompt (wasteful)")

print(f"\n  💡 Same instruction. Verbose = {len(enc.encode(verbose))/len(enc.encode(short)):.1f}x more tokens → {len(enc.encode(verbose))/len(enc.encode(short)):.1f}x more cost!")

# Output:
# Short prompt (optimized):  11 tokens
#   Gemini 2.5 Flash (input)  : $0.000003/call -> ~Rs 83/month   (10k calls/day)
#   GPT-5 (input)             : $0.000014/call -> ~Rs 346/month
#   Claude Sonnet 4.6 (input) : $0.000033/call -> ~Rs 832/month
# Verbose prompt (wasteful): 33 tokens -> 3.0x more tokens = 3.0x more cost

- `PRICES` - a plain lookup of dollars-per-million-tokens for each model (input and output priced separately, because output usually costs more).

- `analyze_cost(text, label, daily_calls)` - the worker. It encodes the text, counts tokens, then for each model does the one bit of arithmetic that runs your bill: `tokens × price ÷ 1,000,000`, scaled to per-day and per-month.

- It runs twice - once on a tight prompt, once on a wordy one that means the same thing - then prints how many times more the wordy version costs.

One formula does all the work: **cost = tokens × price-per-token.** Fewer tokens, smaller bill.

#### 💡 What just happened

⚡ What just happened?
  A verbose prompt ("I would like you to please kindly...", 33 tokens) used **3x more tokens** than a concise prompt ("Summarize: key points only, 3 bullets", 11 tokens) - for the same instruction. At 10K calls/day on GPT-5, that's the difference between about ₹346/month and about ₹1,040/month. **This is why Module 13.2 (Prompt Optimization) teaches you to compress prompts. Tokenization awareness = cost awareness.**

---

## 🎯 Concept 4: Three Tokenizer Families - BPE vs WordPiece vs SentencePiece

### Three Tokenizer Families - BPE vs WordPiece vs SentencePiece

Different models use different tokenization strategies.

| Tokenizer | Algorithm | Used by | Vocab size | Multilingual |
|---|---|---|---|---|
| tiktoken-style BPE | Byte-level Byte-Pair Encoding | GPT-3.5/4/4o, Llama 3+ (128K), Mistral (Tekken) | 100K-200K | Good (newer vocabs better) |
| SentencePiece (BPE/Unigram) | BPE or Unigram | Gemini, T5, Llama 1-2, older Mistral | 32K-256K | Good |
| WordPiece | Greedy longest-match | BERT, DistilBERT | 30K | Moderate |

> ℹ️ **Info**
>
> Which tokenizer matters to you?
> If you're **calling an API** (Gemini, GPT-4): you can't change the tokenizer, but you can optimize your prompts to use fewer tokens. If you're **fine-tuning or self-hosting** (Module 9): tokenizer choice affects vocabulary size, training efficiency, and multilingual performance. SentencePiece is the most versatile - it works directly on raw text without pre-tokenization. **When to use which:** BPE/tiktoken if you call OpenAI, SentencePiece if you self-host multilingual models, WordPiece mainly for BERT-family encoders - the tradeoff is vocabulary size versus how cleanly each handles raw, non-English text.

---

## 🎯 Concept 5: From Tokens to Embeddings - Meaning in Numbers

### From Tokens to Embeddings - Meaning in Numbers

Tokens are just IDs. Embeddings give them meaning - a 768-dimensional "address" in meaning-space.

Token → Embedding: the full pipeline

**📝 `embeddings_demo.py Gemini`** - *API*

In [ ]:
# =============================================
# EMBEDDINGS - From tokens to meaning vectors
# pip install google-genai numpy   (google-genai is the current SDK;
# the old google-generativeai package is deprecated)
# =============================================

from google import genai
from google.genai import types
import numpy as np
import os

# Client reads GEMINI_API_KEY (or GOOGLE_API_KEY) from the environment
client = genai.Client()

def get_embedding(text: str) -> np.ndarray:
    # gemini-embedding-001: 3072 dims by default; 768 is plenty here
    result = client.models.embed_content(
        model="gemini-embedding-001",
        contents=text,
        config=types.EmbedContentConfig(output_dimensionality=768),
    )
    return np.array(result.embeddings[0].values)

def cosine_sim(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

# Get embeddings for related and unrelated words
words = ["king", "queen", "prince", "biryani", "programming", "Python"]
embeddings = {w: get_embedding(w) for w in words}

print("Embedding Similarity Matrix:\n")
print(f"  {'':12s}" + "".join(f"{w:>12s}" for w in words))
for w1 in words:
    row = f"  {w1:12s}"
    for w2 in words:
        sim = cosine_sim(embeddings[w1], embeddings[w2])
        row += f"{sim:12.3f}"
    print(row)

print(f"\n  Embedding dimension: {len(embeddings['king'])}")
print(f"\n  Key observations:")
print(f"  • king ↔ queen:       HIGH (both royalty)")
print(f"  • king ↔ biryani:     LOW (unrelated)")
print(f"  • programming ↔ Python: HIGH (related concepts)")
print(f"  • biryani ↔ Python:   LOW (food vs code)")
print(f"\n  This is how RAG search works (Module 8):")
print(f"  convert query → embedding → find nearest documents.")

# Output: (illustrative - exact cosine values vary by model version)
#               king  queen  prince biryani  prog. Python
#   king        1.00   0.78   0.72   0.31   0.30   0.33
#   queen       0.78   1.00   0.70   0.30   0.29   0.31
#   biryani     0.31   0.30   0.29   1.00   0.28   0.27
#   Embedding dimension: 768
#   king <-> queen: HIGH | king <-> biryani: LOW | programming <-> Python: HIGH

- `genai.Client()` - opens the connection to the embedding API (it reads your key from the environment, never hard-coded).

- `get_embedding(text)` - sends a word to `gemini-embedding-001` and gets back its vector: a list of 768 numbers that encodes the word's meaning.

- `cosine_sim(a, b)` - the closeness score. It measures the *angle* between two vectors (`dot(a,b) ÷ (|a|·|b|)`): 1.0 = same direction (related), near 0 = unrelated.

- The double loop scores every word against every other and prints the grid - exactly how RAG finds relevant documents (Module 8).

Two steps: **turn each word into a vector, then compare vectors by angle.** Close angle = close meaning.

#### 💡 What just happened

⚡ What just happened?
  Each word was converted to a 768-dimensional embedding vector using gemini-embedding-001. "King" and "queen" have high similarity (both royalty). "King" and "biryani" have low similarity (unrelated). "Programming" and "Python" are close (related concepts). **This is the complete pipeline: text → tokens → token IDs → embeddings → meaning. Everything in this course - attention, RAG, similarity search, fine-tuning - operates on these embeddings.** **Tradeoff:** more dimensions capture more nuance but cost more storage and slower search - we quantify exactly that in Step 8.

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 6: Why LLMs Can't Count Letters - Tokenization Quirks

### Why LLMs Can't Count Letters - Tokenization Quirks

Many bizarre LLM failures trace back to tokenization, not model intelligence.

> 🍓 **Analogy**
>
> **The "Strawberry Problem":** Ask GPT-4 "How many r's in strawberry?" - it often says 2 (correct answer: 3). Why? Because "strawberry" is tokenized as `["st", "raw", "berry"]` (3 subword chunks, not 10 letters) - the model never sees individual letters. It's operating on subword chunks, not characters. *This is not a model intelligence failure - it's a tokenization limitation.*
> **Try it in 2026:** the newest reasoning flagships - Claude's latest (claude-opus-4-8) and Gemini - usually answer **3** correctly, while GPT-5.x still tends to trip. But a correct answer does **not** mean tokenization was fixed: `strawberry` is still `['st', 'raw', 'berry']` underneath. These models compensate by reasoning the letters out (and by having seen this exact famous question countless times in training). Proof the limit remains: ask about a **rare or made-up word** and even strong models wobble. Reasoning routes around the tokenization limit - it does not erase it.

**📝 `tokenization_quirks.py Why LLMs`** - *Fail*

In [ ]:
# =============================================
# TOKENIZATION QUIRKS - Why LLMs struggle with
# letter counting, reversal, and arithmetic
# =============================================

import tiktoken

enc = tiktoken.encoding_for_model("gpt-4o")

# ── 1. The Strawberry Problem ──
word = "strawberry"
tokens = enc.encode(word)
decoded = [enc.decode([t]) for t in tokens]
print(f"'strawberry' → {decoded}")
print(f"Model sees {len(tokens)} chunks, NOT 10 letters")
print(f"Actual r count: {word.count('r')} - model often says 2\n")

# ── 2. String Reversal ──
word = "hello"
tokens = enc.encode(word)
decoded = [enc.decode([t]) for t in tokens]
print(f"'hello' → {decoded}")
print(f"Model can't easily reverse within merged tokens\n")

# ── 3. Arithmetic Inconsistency ──
for num in ["234", "2345", "23456", "1000000"]:
    tokens = enc.encode(num)
    decoded = [enc.decode([t]) for t in tokens]
    print(f"  '{num}' → {decoded} ({len(tokens)} tokens)")
print(f"Numbers tokenize inconsistently - that's why LLM math fails\n")

# ── 4. The Multilingual "Token Tax" ──
texts = {
    "English":  "AI is transforming how we learn and work.",
    "Hindi":    "कृत्रिम बुद्धिमत्ता हमारे सीखने और काम करने के तरीके को बदल रही है।",
    "Telugu":   "కృత్రిమ మేధస్సు మనం నేర్చుకునే విధానాన్ని మారుస్తోంది.",
}
en_count = len(enc.encode(texts["English"]))
for lang, text in texts.items():
    count = len(enc.encode(text))
    ratio = count / en_count
    print(f"  {lang:<10s} {count:>3} tokens ({ratio:.1f}x English)")
print(f"\n  Hindi costs ~{len(enc.encode(texts['Hindi']))/en_count:.1f}x more → same meaning, higher API bill!")
print(f"  Token Tax (arXiv 2509.05486): Arabic/Hindi/Telugu run ~3-5x English")

# Output:
# 'strawberry' -> ['st', 'raw', 'berry']   (3 chunks, NOT 10 letters)
# Actual r count: 3   (the model often says 2)
# 'hello' -> ['hello']
# '234' -> ['234'] (1); '2345' -> ['234','5'] (2); '1000000' -> ['100','000','0'] (3)
# English 1.0x | Hindi 2.2x | Telugu 2.4x English (o200k)

- The same trick powers all four demos: `enc.encode(x)` turns text into token IDs, then `[enc.decode([t]) for t in tokens]` turns each ID back into its text - so you can **see the exact chunks the model sees**.

- **Strawberry** - decoding shows `['st','raw','berry']`: three chunks, never the 10 letters, so the model cannot count r's it never sees.

- **Numbers** - the loop shows the same digits splitting differently (`234` = 1 token, `23456` = 2), which is why arithmetic is shaky.

- **Languages** - the last loop counts tokens per language and divides by English, surfacing the 2-5x "token tax".

One technique - decode each token - exposes the root of every quirk: **the model reads chunks, not characters.**

#### 💡 What just happened

⚡ What just happened?
  Four tokenization quirks that affect production: (1) **Letter counting fails** because the model sees subword tokens, not characters - use code execution tools instead. (2) **String reversal breaks** across token boundaries. (3) **Arithmetic is inconsistent** because numbers tokenize differently (234 = 1 token, 23456 = 2 tokens). (4) **The "Token Tax"** - Hindi/Telugu cost roughly 2-3x more tokens than English for the same meaning, and the systematic study ([arXiv 2509.05486](https://arxiv.org/abs/2509.05486)) finds heavy-script languages running 3-5x. **Karpathy's key insight: "A lot of weird behaviors of LLMs actually trace back to tokenization." Solutions: code execution tools (Module 10), token-aware prompt design (Module 5), multilingual-optimized models (Module 13).**

> 📦 **Info**
>
> 🔮 Frontier: is tokenization on the way out?
> Every quirk in this step - the strawberry miscount, the token tax, brittle arithmetic - traces to one root: the model never sees raw characters, only subword tokens. So researchers asked the obvious question - what if we drop tokenization entirely?
> Meta's **Byte Latent Transformer (BLT)** does exactly that. Instead of a fixed vocabulary it reads raw **bytes** and groups them into *patches* on the fly, spending more compute where the text is less predictable (high entropy). It matches Llama 3 quality with up to roughly 50% lower inference cost and - because it sees bytes - it is far better at spelling, noisy input, and low-resource languages. Tokenization is not dead, but byte-level models are its most serious challenger in years. [Read the BLT paper (arXiv 2412.09871)](https://arxiv.org/abs/2412.09871).

---

## 🎯 Concept 7: Special Tokens & Chat Templates - The Hidden Control Layer

### Special Tokens & Chat Templates - The Hidden Control Layer

Special tokens control when the model starts, stops, and how it distinguishes system/user/assistant messages.

**📝 `special_tokens.py Control`** - *Layer*

In [ ]:
# =============================================
# SPECIAL TOKENS - The hidden control signals
# =============================================

# Every model has reserved token IDs with special roles:
special_tokens = {
    "[BOS] / <s>":         "Beginning of sequence - signals start",
    "[EOS] / </s>":        "End of sequence - model stops generating",
    "[PAD]":               "Padding - fills short sequences in batches",
    "[UNK]":               "Unknown - fallback for missing vocab",
    "<|endoftext|>":       "OpenAI's document separator",
    "<|im_start|>":        "Chat message boundary (OpenAI)",
    "[INST] / [/INST]":    "Instruction boundaries (Mistral/Llama 2)",
    "<|start_header_id|>": "Role marker (Llama 3)",
}

print("SPECIAL TOKENS ACROSS MODELS:\n")
for token, role in special_tokens.items():
    print(f"  {token:<25s} {role}")

# Chat template example - how models know who's talking
print(f"""
CHAT TEMPLATES - Different models, different formats:

OpenAI (GPT-4):
  <|im_start|>system
  You are a helpful assistant.<|im_end|>
  <|im_start|>user
  What is GenAI?<|im_end|>

Llama 3:
  <|begin_of_text|><|start_header_id|>system<|end_header_id|>
  You are a helpful assistant.<|eot_id|>

Mistral:
  [INST] What is GenAI? [/INST]

Why this matters:
  - Wrong template during fine-tuning = broken model
  - Missing EOS token = model generates forever
  - Double BOS tokens = confused predictions
  - Module 9 (fine-tuning) requires getting this exactly right
""")

# Output:
# SPECIAL TOKENS ACROSS MODELS:
#   [BOS] / <s>            Beginning of sequence - signals start
#   [EOS] / </s>           End of sequence - model stops generating
#   <|endoftext|>          OpenAI's document separator
#   <|im_start|>           Chat message boundary (OpenAI)
#   [INST] / [/INST]       Instruction boundaries (Mistral/Llama 2)
#   <|start_header_id|>    Role marker (Llama 3)
# ...followed by the OpenAI / Llama 3 / Mistral chat-template formats.

#### 💡 What just happened

⚡ What just happened?
  Special tokens are the **traffic signals** of LLM generation. [BOS] says "start here." [EOS] says "stop now." Chat templates wrap messages so the model knows who's speaking. **Each model family uses different tokens** - Llama 3 uses header tags, Mistral uses [INST], OpenAI uses im_start. Getting these wrong during fine-tuning (Module 9) is one of the most common bugs. The tokenizer handles these automatically during inference, but you must understand them for custom training. **When it matters:** at inference the SDK fills the template in for you; the limitation surfaces in fine-tuning, where the wrong template silently breaks training.

---

## 🎯 Concept 8: The Embedding Landscape - From Word2Vec to Matryoshka

### The Embedding Landscape - From Word2Vec to Matryoshka

Embeddings evolved from static word vectors to dynamic, dimension-adaptive representations that power RAG.

> 📍 **Analogy**
>
> **Embedding evolution in 30 seconds:** Word2Vec (2013) gave "bank" ONE vector - same for river bank and financial bank. ELMo (2018) gave it different vectors based on context. BERT/GPT embeddings are fully contextual - every token's vector depends on ALL surrounding tokens. Modern embedding APIs (gemini-embedding-001, OpenAI text-embedding-3) package this into a single call. *"King - Man + Woman = Queen"* - this analogy from Word2Vec is still the best explanation of what embeddings capture.

**📝 `embedding_landscape.py 2026`** - *Models*

In [ ]:
# =============================================
# EMBEDDING LANDSCAPE - Models, Dimensions, RAG
# =============================================

# 2026 embedding models (MTEB leaderboard, approximate - check current rates)
models = [
    ("gemini-embedding-001",   3072,  "~$0.15/M", "Google",   "~#1 English MTEB; Matryoshka 3072/1536/768"),
    ("Qwen3-Embedding-8B",     4096,  "open",     "Alibaba",  "~#1 multilingual MTEB; self-host, instruct"),
    ("text-embedding-3-large", 3072,  "$0.13/M",  "OpenAI",   "Matryoshka: truncate 256-3072"),
    ("text-embedding-3-small", 1536,  "$0.02/M",  "OpenAI",   "Best value for most RAG apps"),
    ("voyage-3.5",             1024,  "~$0.06/M", "Voyage",   "Strong retrieval, good price/quality"),
    ("BGE-M3",                1024,  "open",     "BAAI",     "Open-source, multilingual, long-context"),
]

print(f"{'Model':<26s} {'Dims':>5} {'Cost':>10} {'Provider':<10} Notes")
print("-" * 90)
for name, dims, cost, provider, notes in models:
    print(f"  {name:<24s} {dims:>5} {cost:>10} {provider:<10} {notes}")

# Dimension vs Storage Cost
print(f"""
DIMENSION TRADE-OFFS (10M vectors, approximate):
  256 dims  → ~10 GB  → ~$1.25/month → ~95% of max accuracy
  768 dims  → ~30 GB  → ~$3.75/month → ~98% of max accuracy
  3072 dims → ~120 GB → ~$15/month   → 100% accuracy

Matryoshka embeddings (gemini-embedding-001, text-embedding-3):
  Train once at full size, truncate to ANY size at query time.
  A 768-dim slice keeps ~98% of the quality of the full 3072 dims.
  → Use 768 for most RAG apps, save ~75% storage vs 3072.

For Module 8 (RAG): embedding quality = retrieval quality = answer quality.
  Sensible defaults: gemini-embedding-001 (768) or text-embedding-3-small.
""")

# Output:
# Model                   Dims      Cost  Provider  Notes
# gemini-embedding-001    3072  ~$0.15/M  Google    ~#1 English MTEB; Matryoshka
# Qwen3-Embedding-8B      4096      open  Alibaba   ~#1 multilingual MTEB; self-host
# text-embedding-3-large  3072   $0.13/M  OpenAI    Matryoshka 256-3072
# text-embedding-3-small  1536   $0.02/M  OpenAI    Best value for RAG
# voyage-3.5              1024  ~$0.06/M  Voyage    Strong retrieval
# BGE-M3                  1024      open  BAAI      Open-source, multilingual

#### 💡 What just happened

⚡ What just happened?
  The embedding landscape in 2026: **gemini-embedding-001** (Matryoshka 3072/1536/768, roughly $0.15/M, around #1 on the English MTEB leaderboard), **Qwen3-Embedding-8B** (open-source, around #1 on multilingual MTEB), **text-embedding-3** (OpenAI, Matryoshka), and self-host options like **BGE-M3**. **Key insight: 768 dimensions is enough for most RAG.** Going from 768 to 3072 adds roughly 4x storage for only about 2% more accuracy. Matryoshka embeddings let you train once and truncate - 256d for fast search, 768d for accurate search. **You will build a complete RAG pipeline on these embeddings in Module 8.**

A team prices their app by **counting words**: "about 750 words per request, so ~750 tokens." This is the wrong way, and it breaks when real text arrives - 750 English words is closer to 1,000 tokens, and 750 words of Hindi can be 2,500+. **Do not estimate cost from word or character counts.** Always measure with the real tokenizer (tiktoken, or the provider's tokenizer endpoint) before you trust a budget.

### 🧮 Putting it together

Text becomes numbers in two moves. **Tokenization** chops text into subword tokens by frequency (BPE) - which is why your bill, your context window, the "token tax" on Indian languages, and the strawberry miscount all trace back to the same place. **Embeddings** then turn each token ID into a vector whose direction encodes meaning, so "king" and "queen" land close together.

You will use both halves constantly: you will build a retrieval pipeline on these embeddings in Module 8, wrestle with chat templates again in Module 9 (fine-tuning), and revisit token economics in Module 13 (cost). And you saw the frontier - byte-level models like BLT that may one day make tokenization optional.

## Key Takeaways

> ℹ️ **Info**
>
> What you learned - 8 concepts
> - **Tokenization** = how AI cuts text into subword pieces. "unbelievable" → ["un", "believ", "able"]. Not characters, not words.
> - **BPE** = the algorithm behind GPT/Llama tokenizers. Start from 256 bytes, merge frequent pairs, build vocabulary bottom-up.
> - **Three families** = tiktoken/BPE (GPT, Llama 3+), SentencePiece (Gemini, T5), WordPiece (BERT). Different approaches, same goal.
> - **Token economics** = more tokens = more cost. Gemini 2.5 Flash is roughly 8-10x cheaper than GPT-5 or Claude Sonnet on input (2026). Output tokens cost ~2-5x more than input.
> - **Embeddings** = tokens become 768-dimensional vectors that encode meaning. Similar words = nearby vectors.
> - **Tokenization quirks** = why LLMs fail at letter counting ("strawberry"), string reversal, and arithmetic. All trace to subword tokenization.
> - **Special tokens** = [BOS], [EOS], chat templates. Different per model. Wrong template = broken fine-tuning.
> - **Embedding landscape** = gemini-embedding-001 and Qwen3-Embedding top MTEB; OpenAI text-embedding-3 is Matryoshka. 768 dims is enough for most RAG.

## Tokenization → Production Impact

| Concept | Production Impact | Where You'll Use It |
|---|---|---|
| BPE Algorithm | Understand why models tokenize differently | Module 9 (custom tokenizer for fine-tuning) |
| Token Economics | ~8-10x cost difference between models | Module 13 (cost optimization, model routing) |
| Multilingual Token Tax | Hindi costs ~2-3x more than English | Module 13 (multilingual app budgeting) |
| Tokenization Quirks | Explains letter counting, math failures | Module 10 (code execution tools as fix) |
| Special Tokens | Wrong template = broken fine-tuning | Module 9 (LoRA, chat template setup) |
| Embedding Models | Embedding quality = RAG quality | Module 8 (vector search, chunking) |
| Matryoshka Dimensions | 768 dims saves ~75% storage vs 3072 | Module 8 (RAG cost optimization) |
| Prompt Token Awareness | Token-efficient prompts save ~30-60% | Module 5 (prompt engineering) |

*Practice exercises are in the companion practice notebook.*

---

## 🎓 Summary

You've completed the practical part of **Tokenization & Embeddings- How AI Reads Text**.

### Next steps:
1. Re-run cells with different parameters to build intuition
2. Try the practice exercises (see `practice-lab/practice-lab-4_1.ipynb` if available)
3. Review the full HTML lesson for the complete narrative

*Generated from `lesson-4.1.html` - regenerate this notebook whenever the source HTML is updated.*
